# Lesson 91 · 🎨 Attention illustrated — seeing where the model "looks": self-attention weight heatmaps and LoRA's low-rank structure

**Learning goals**
1. Read the diagonal and off-diagonal signals in a single-head self-attention heatmap.
2. Learn to pull the top-k attended token pairs out of an attention matrix.
3. Understand LoRA's low-rank decomposition and how much it saves in parameters.
4. Connect the attention-weight and LoRA low-rank-structure plots to the implementations in L83 / L84.

> Look at the diagonal first, then off the diagonal: the first tells you a word is looking at itself, the second that it's looking at other words.

**Goal**: visualize the token attention patterns of a single-head Transformer attention, and draw LoRA's parameter savings.

🔗 Aurora connection: `aurora.llm` (KVCache, sampling, TF-IDF retrieval) · visual archive (aurora.whisper isn't implemented yet — self-attention visualization will live in the aurora.multimodal module)

← **Previous lesson** · [L90 · Conversational RAG and a tool-using agent (ReAct)](L90_agent.ipynb)

> Last lesson covered **conversational RAG and a tool-using agent (ReAct)**: conversation memory, source attribution, a podcast Q&A pipeline, and a tool-using agent running the Thought → Action → Observation loop.
> This lesson explores **attention illustrated**.

## When a neural network reads a sentence, where is it "looking"?

When you read "the **guitar** is an **instrument**", your brain knows "guitar" and "instrument" are the most closely related pair of words — the first is a specific example of the second.

**How does a Transformer manage this?** With self-attention: every word assigns an "attention weight" to every other word.

```
Q = X @ W_Q   # Query: what am I asking about?
K = X @ W_K   # Key: what are the other words?
A = softmax(Q @ Kᵀ / √d_k)   # weight matrix: who am I paying the most attention to?
```

The result is a `(seq_len × seq_len)` weight matrix that can be drawn directly as a heatmap — the horizontal axis is "the word being attended to", the vertical axis is "the word doing the attending".

**The link to LoRA**: LoRA freezes the weight matrices W (including W_Q and W_K) and trains only the low-rank update ΔW=B@A. The smaller the rank r → the fewer the parameters, but also the more limited the range over which the attention weights can be adjusted.

In this lesson: draw heatmaps to get an intuitive feel for attention patterns, then implement an analysis function `find_top_pairs()` that pulls the strongest token associations out of the weight matrix.

## How do you read a heatmap?

**A bright diagonal** (`A[i,i]` high) → words mostly attend to themselves (common in shallow layers)

**Bright off-diagonal cells** (`A[i,j]` high, i≠j) → word i depends strongly on the meaning of word j (grammatical/semantic links in a language model)

**Randomly initialized W_Q, W_K** carry no meaning, so the heatmap shows a random pattern; after fine-tuning, the weights of semantically related word pairs rise markedly.

**What √d_k is for**: without it, high-dimensional dot products have a large variance → softmax becomes extremely sharp → gradients all but vanish; dividing by √d_k normalizes the variance to 1.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The attention heatmap

The output weight matrix `A` of self-attention has shape `(seq_len, seq_len)`;
element `A[i, j]` says how much token `i` attends to token `j`.

How it's computed:
```
Q = X @ W_Q   # (seq, d_k)
K = X @ W_K   # (seq, d_k)
A = softmax(Q @ K.T / sqrt(d_k))   # (seq, seq)
```

In the heatmap the horizontal axis is the Key (the token being attended to) and the vertical axis is the Query (the token doing the attending).
High weights on the diagonal mean tokens mostly attend to themselves; bright spots off the diagonal reveal semantic links.

In [ ]:
def scaled_dot_product_attention(X, W_Q, W_K, d_k):
    """Compute the self-attention weight matrix by hand"""
    Q = X @ W_Q
    K = X @ W_K
    scores = Q @ K.T / (d_k ** 0.5)
    scores_exp = np.exp(scores - scores.max(axis=-1, keepdims=True))
    A = scores_exp / scores_exp.sum(axis=-1, keepdims=True)
    return A

np.random.seed(42)
tokens = ['the', 'guitar', 'is', 'an', 'instrument']
seq_len, d_model, d_k = 5, 16, 8
X = np.random.randn(seq_len, d_model)
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)

A = scaled_dot_product_attention(X, W_Q, W_K, d_k)
print('Attention matrix shape:', A.shape)
print('Row sums (should be 1.0):', A.sum(axis=1).round(4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A, cmap='Blues', vmin=0, vmax=A.max())
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_xlabel('Key (attended to)', fontsize=11)
ax.set_ylabel('Query (attending)', fontsize=11)
ax.set_title('Self-attention heatmap\n"the guitar is an instrument"', fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{A[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if A[i,j] > 0.35 else 'black')
plt.tight_layout()
plt.show()
print(f'guitar→instrument: {A[1,4]:.4f}')
print(f'guitar→guitar:     {A[1,1]:.4f}')

## 2. A bar chart of LoRA parameter counts

LoRA freezes the original weights `W` (shape `d×d`) and trains only two low-rank matrices:
```
A: (r, d)   parameter count = r * d
B: (d, r)   parameter count = d * r
ΔW = B @ A  # the effective shape is still (d, d), but with only 2*d*r degrees of freedom
```
The saving = `1 - 2r/d`. With d=4096 (LLaMA-7B / GPT-3 scale) and r=8, the saving is > 99.6%.

The bar chart compares, at a glance, the trainable parameters at different ranks against the parameters of the original matrix.

In [ ]:
d = 4096
ranks = [1, 2, 4, 8, 16, 32, 64]
total_params = d * d
lora_params = [2 * r * d for r in ranks]
savings_pct = [(1 - lp / total_params) * 100 for lp in lora_params]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
x = range(len(ranks))
bars = ax.bar(x, lora_params, color='steelblue', alpha=0.85, label='LoRA trainable parameters')
ax.axhline(total_params, color='tomato', linewidth=2, linestyle='--',
           label=f'original matrix ({total_params:,})')
ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels([f'r={r}' for r in ranks])
ax.set_ylabel('parameter count (log scale)')
ax.set_title(f'LoRA parameters vs the original matrix\n(d={d})')
ax.legend()
for bar, lp in zip(bars, lora_params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.4,
            f'{lp:,}', ha='center', va='bottom', fontsize=8)

ax2 = axes[1]
ax2.bar(x, savings_pct, color='seagreen', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels([f'r={r}' for r in ranks])
ax2.set_ylabel('parameter saving (%)')
ax2.set_ylim(95, 100.5)
ax2.set_title('LoRA saving (d=4096)')
for i, sp in enumerate(savings_pct):
    ax2.text(i, sp + 0.05, f'{sp:.2f}%', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()
print(f'r=8 saving: {savings_pct[3]:.2f}%  trainable: {lora_params[3]:,} / {total_params:,}')

## Parameter experiment: the guitar-instrument attention weight

**Sentence**: `"the guitar is an instrument"`

**Knobs**: `exp_d_k` (the attention key dimension), `exp_seed` (the random seed)

**What to expect**: randomly initialized W_Q / W_K carry no meaning, so the `guitar→instrument` weight
isn't necessarily higher than other token pairs. As you change `exp_d_k`:
- smaller `d_k` → larger variance in the softmax input → a sharper distribution (one position's weight close to 1)
- larger `d_k` → the input is divided by a larger `sqrt(d_k)` → a flatter distribution (close to uniform)
In the weights of a fine-tuned language model, the attention weights of semantically related token pairs are markedly higher than those of unrelated pairs.

In [ ]:
# Change exp_d_k or exp_seed and watch how the guitar→instrument weight changes
exp_d_k = 4       # try 4 / 8 / 32 / 64
exp_seed = 0      # change the seed to see the random fluctuation

np.random.seed(exp_seed)
X_exp = np.random.randn(seq_len, d_model)
W_Q_exp = np.random.randn(d_model, exp_d_k)
W_K_exp = np.random.randn(d_model, exp_d_k)
A_exp = scaled_dot_product_attention(X_exp, W_Q_exp, W_K_exp, exp_d_k)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(A_exp, cmap='Blues', vmin=0, vmax=A_exp.max())
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens, fontsize=11)
ax.set_yticklabels(tokens, fontsize=11)
ax.set_title(f'Attention heatmap d_k={exp_d_k}, seed={exp_seed}', fontsize=12)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{A_exp[i,j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if A_exp[i,j] > 0.35 else 'black')
plt.tight_layout()
plt.show()

guitar_idx, instrument_idx = 1, 4
print(f'd_k={exp_d_k}: guitar→instrument = {A_exp[guitar_idx, instrument_idx]:.4f}')
print(f'd_k={exp_d_k}: guitar→guitar     = {A_exp[guitar_idx, guitar_idx]:.4f}')
print('Tip: the larger d_k, the flatter the softmax distribution; the smaller, the sharper')

## ✏️ Exercise: `find_top_pairs(A, tokens, k=3)`

**Task**: from a `(seq, seq)` attention-weight matrix, pull out the k (query, key) token pairs with the highest attention weights.

**Signature**:
```python
def find_top_pairs(A: np.ndarray, tokens: list, k: int = 3) -> list[tuple]:
    # returns: [(query_token, key_token, weight), ...] in descending order of weight
```

**Three-step route**:

| Step | What to do | Code hint |
|---|---|---|
| 1 | List every (i,j) pair with its weight | `np.ndindex(A.shape)` or a double loop |
| 2 | Sort | `sorted(..., key=lambda x: -x[2])` |
| 3 | Take the first k and map index→token | `tokens[i], tokens[j], float(A[i,j])` |

**Acceptance criteria**:
- the returned list has exactly k items
- the first item has the highest weight (descending order)
- each item has the form `(str, str, float)`

In [ ]:
def find_top_pairs(A: np.ndarray, tokens: list, k: int = 3) -> list:
    """Pull the strongest token-association pairs out of an attention-weight matrix.

    Returns
    -------
    list of (query_token: str, key_token: str, weight: float)
        In descending order of weight, length k.
    """
    # TODO: go through every (i,j) pair, sort by A[i,j] in descending order, take the first k
    raise NotImplementedError("please implement find_top_pairs")


# ── Check ──────────────────────────────────────────────────────────────────────
try:
    top_pairs = find_top_pairs(A, tokens, k=3)
    assert len(top_pairs) == 3, f"should return 3 pairs, got {len(top_pairs)}"
    assert all(len(p) == 3 for p in top_pairs), "each item should be a (query, key, weight) triple"
    q, k_tok, w = top_pairs[0]
    assert isinstance(q, str) and isinstance(k_tok, str), "the first two items should be strings"
    assert isinstance(w, float), "the third item should be a float weight"
    assert w >= top_pairs[1][2] >= top_pairs[2][2], "should be sorted by weight in descending order"
    print(f"✅ find_top_pairs check passed; the strongest associations:")
    for q_tok, k_tok, weight in top_pairs:
        print(f"   {q_tok:10s} → {k_tok:10s}  weight {weight:.4f}")
except (NotImplementedError, TypeError):
    print("⚠️ find_top_pairs isn't implemented yet")

## ✏️ Closed-book derivation check cell — the maths of attention

**Rule: close every cell above and work it out from memory.**

**Problem**: sequence length seq=3, d_k=4, input matrix X of shape (3, 16).

1. What is the shape of Q = X @ W_Q? (W_Q: 16×4)
2. What is the shape of scores = Q @ Kᵀ?
3. After dividing by √d_k, if scores[0]=[2.0, 1.0, 0.5], write out the steps of the softmax computation (including the numerical-stability trick)
4. What should A.sum(axis=-1) be? (for each row)
5. The LoRA saving formula: with d=512 and r=8, what are the original matrix's parameter count vs LoRA's parameter count vs the saving?

(Write your derivation here...)

In [ ]:
# Check: the attention maths worked out by hand
import numpy as np

# 1-2. shape verification
_seq, _d_model, _d_k = 3, 16, 4
_X = np.random.randn(_seq, _d_model)
_W_Q = np.random.randn(_d_model, _d_k)
_W_K = np.random.randn(_d_model, _d_k)
_Q = _X @ _W_Q
_K = _X @ _W_K
assert _Q.shape == (_seq, _d_k), f"Q shape should be ({_seq},{_d_k}), got {_Q.shape}"
print(f"1 ✅  Q = X @ W_Q: ({_seq},{_d_model}) @ ({_d_model},{_d_k}) = ({_seq},{_d_k})")

_scores = _Q @ _K.T
assert _scores.shape == (_seq, _seq), f"scores shape should be ({_seq},{_seq}), got {_scores.shape}"
print(f"2 ✅  scores = Q @ Kᵀ: ({_seq},{_d_k}) @ ({_d_k},{_seq}) = ({_seq},{_seq})")

# 3. softmax calculation
_raw = np.array([2.0, 1.0, 0.5])
_stable = _raw - _raw.max()
_exp = np.exp(_stable)
_probs = _exp / _exp.sum()
assert abs(_probs.sum() - 1.0) < 1e-9
print(f"3 ✅  softmax([2,1,0.5]): stable={_stable}, exp={_exp.round(4)}, probs={_probs.round(4)}")

# 4. row sums
_A_test = scaled_dot_product_attention(_X, _W_Q, _W_K, _d_k)
assert np.allclose(_A_test.sum(axis=-1), 1.0, atol=1e-9), "every row should sum to 1"
print(f"4 ✅  A.sum(axis=-1) is all 1.0 (guaranteed by softmax normalization)")

# 5. LoRA savings
d_lora, r_lora = 512, 8
orig = d_lora * d_lora
lora_p = 2 * d_lora * r_lora
saving = (1 - lora_p / orig) * 100
assert abs(saving - (1 - 2*r_lora/d_lora)*100) < 1e-9
print(f"5 ✅  d={d_lora},r={r_lora}: original={orig:,}, LoRA={lora_p:,}, saving={saving:.2f}%")

## Wrap-up

The `(seq, seq)` weight matrix output by `scaled_dot_product_attention` can be visualized directly as a heatmap, revealing how attention is distributed between tokens in each layer.
LoRA's parameter savings come from the low-rank decomposition `ΔW = B @ A`, and the rank r sets the number of trainable parameters, `2·d·r`; both of these core quantities will go into the fine-tuning module of `aurora.llm`.
That brings the LLM-applications module to a close.

In [ ]:
# ✏️ Self-review for this lesson
l91_review = {
    "attention_heatmap_reading":  None,  # can read a heatmap: diagonal/off-diagonal, semantic-link patterns? True/False
    "sqrt_dk_rationale":          None,  # understand √d_k: stops high-dimensional dot products' large variance → softmax saturation? True/False
    "find_top_pairs_impl":        None,  # find_top_pairs() is correct and the 3-pair check passes? True/False
    "lora_savings_formula":       None,  # can work out LoRA's saving in your head: 2·d·r vs d², saving = (1-2r/d)×100%? True/False
    "whiteboard_passed":          None,  # cleared the whiteboard challenge of deriving the attention maths? True/False
}

unfilled = [k for k, v in l91_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l91_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L91 all clear! The LLM module is done — on to L92: the end-to-end pipeline')

---

→ **Next lesson** · [L92 · The end-to-end pipeline](../10_integration/L92_pipeline.ipynb)

> Next lesson covers **the end-to-end pipeline**: microphone → ASR → LLM → text answer, the whole chain assembled.